In [ ]:
# POS-Parts of Speech and NER-Named entity Recognition practical

In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import re
import pandas as pd
import matplotlib.pyplot as plt


/opt/anaconda3/envs/nlp_course_env/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (None) doesn't match a supported version!
  warnings.warn(


In [2]:
bbc_data=pd.read_csv("bbc_news.csv")
bbc_data.head(3)

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?a...,With much of the UK enduring another period of...
1,1,9267,'Liz Truss the Brief?' World reacts to UK poli...,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_m...,The UK's political chaos has been watched arou...
2,2,7387,Rationing energy is nothing new for off-grid c...,"Wed, 31 Aug 2022 05:20:18 GMT",https://www.bbc.co.uk/news/uk-scotland-highlan...,https://www.bbc.co.uk/news/uk-scotland-highlan...,Scoraig in the north west Highlands has long h...


In [3]:
bbc_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   index        1000 non-null   int64 
 2   title        1000 non-null   object
 3   pubDate      1000 non-null   object
 4   guid         1000 non-null   object
 5   link         1000 non-null   object
 6   description  1000 non-null   object
dtypes: int64(2), object(5)
memory usage: 54.8+ KB


In [4]:
titles=pd.DataFrame(bbc_data["title"])
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


In [6]:
titles["lowercase"]=titles["title"].str.lower()

In [7]:
titles.head()


,title,lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...


In [8]:
en_stopword=stopwords.words("english")

In [10]:
titles["no_stopword"]=titles["lowercase"].apply(lambda x: " ".join([word for word in x.split() if word not in en_stopword]))

In [12]:
titles["no_stopword_no_punct"]= titles.apply(lambda x:re.sub("[^\w\s]","",x["no_stopword"]),axis=1)

In [ ]:
titles.head()

In [20]:
titles["tokenes_raw"]=titles.apply(lambda x:word_tokenize(x["title"]),axis=1)

In [21]:
titles["tokens_clean"]=titles.apply(lambda x:word_tokenize(x["no_stopword_no_punct"]),axis=1)

In [22]:
lemmatizer=WordNetLemmatizer()

In [24]:
titles["tokens_clened_lemmatized"]=titles["tokens_clean"].apply(lambda tokens:[lemmatizer.lemmatize(token) for token in tokens])

In [ ]:
titles.head()

In [27]:
tokens_raw_list=sum(titles["tokenes_raw"],[])

In [28]:
tokens_clean_list=sum(titles["tokens_clean"],[])

In [ ]:
print(tokens_raw_list)

In [35]:
nlp=spacy.load("en_core_web_sm")
spacy_doc=nlp(" ".join(tokens_raw_list))

In [37]:
pos_df=pd.DataFrame(columns=["token","pos_tag"])
for token in spacy_doc:
    pos_df=pd.concat([pos_df,pd.DataFrame.from_records([{"token":token.text,"pos_tag":token.pos_}])],ignore_index=True)

In [ ]:
pos_df.head()

In [45]:
pos_df_counts=pos_df.groupby(["token","pos_tag"]).size().reset_index(name="counts").sort_values(by="counts",ascending=False)

In [48]:
nouns=pos_df_counts[pos_df_counts.pos_tag=="NOUN"][:10]
nouns

,token,pos_tag,counts
4267,war,NOUN,35
3552,record,NOUN,15
3416,police,NOUN,14
4356,year,NOUN,14
4316,win,NOUN,14
3061,living,NOUN,13
4009,tax,NOUN,13
2326,day,NOUN,12
3368,people,NOUN,12
2031,boss,NOUN,11


In [49]:
verbs=pos_df_counts[pos_df_counts.pos_tag=="VERB"][:10]
verbs

,token,pos_tag,counts
3687,says,VERB,30
9,',VERB,14
2670,found,VERB,13
4317,win,VERB,12
4324,wins,VERB,10
2713,get,VERB,9
2388,dies,VERB,9
3990,take,VERB,8
2982,killed,VERB,8
3686,say,VERB,8


In [50]:
adj=pos_df_counts[pos_df_counts.pos_tag=="ADJ"][:10]
adj

,token,pos_tag,counts
3244,new,ADJ,28
1400,Russian,ADJ,21
2606,final,ADJ,16
19,-,ADJ,14
2625,first,ADJ,12
3199,more,ADJ,10
1994,big,ADJ,9
2835,high,ADJ,9
3000,last,ADJ,8
3304,other,ADJ,8


In [54]:
ner_df=pd.DataFrame(columns=["token","ner_tag"])
for token in spacy_doc.ents:
    if pd.isna(token.label_) is False:
        ner_df=pd.concat([ner_df,pd.DataFrame.from_records([{"token":token.text,"ner_tag":token.label_}])],ignore_index=True)

In [55]:
ner_df.head()

,token,ner_tag
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP


In [56]:
ner_df_counts=ner_df.groupby(["token","ner_tag"]).size().reset_index(name="counts").sort_values(by="counts",ascending=False)

In [57]:
ner_df_counts.head()


,token,ner_tag,counts
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19
